In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import h5py, os, yaml, pickle
import numpy as np
import healpy as hp
import matplotlib.pyplot as plt
from tqdm import tqdm
from trianglechain import TriangleChain

import tensorflow as tf
tf.config.experimental.set_memory_growth(tf.config.list_physical_devices(device_type="GPU")[0], True)

from deep_lss.models.base_model import BaseModel
from deep_lss.utils import configuration
from deep_lss.nets import NETWORKS

from msi.utils import preprocessing

from msfm.utils import prior, parameters, files, logger, observation

lensing, clustering, combined = False, False, False

2025-09-16 02:15:14.694684: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-09-16 02:15:14.694706: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-09-16 02:15:14.695781: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-09-16 02:15:14.702232: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-09-16 02:15:16.737905: W tensorflow/compiler/tf2

# v13

### clustering

In [3]:
# # # wandering-shadow-1084 (https://wandb.ai/eth-cosmo/y3-deep-lss/runs/z9to945a/overview)
# # # even more conservative run, 32mpc/h smoothing with 10% noise -> l_max = [82, 121, 158, 189]
# # model_dir = "/pscratch/sd/a/athomsen/run_files/v13/extended/clustering/mutual_info/2025-01-16_00-50-58_deepsphere_default"
# # # n_steps = 80_000
# # # n_steps = 140_000
# # # n_steps = 120_000
# # n_steps = 170_000

# params = ["Om", "s8", "w0", "bg1", "bg2", "bg3", "bg4"]
# n_z = 4
# clustering = True

# v14

### lensing

In [4]:
# # honest-haze-1091 (https://wandb.ai/eth-cosmo/y3-deep-lss/runs/58tev0dz/overview)
# # first v14 run
# model_dir = "/pscratch/sd/a/athomsen/run_files/v14/extended/lensing/mutual_info/2025-04-19_18-54-31_deepsphere_default"
# # n_steps = 100_000
# # n_steps = 200_000
# # n_steps = 300_000
# n_steps = 400_000

# checkpoint_number = n_steps // 10_000
# params = ["Om", "s8", "w0", "Aia", "n_Aia", "bta"]
# n_z = 4
# lensing = True

### clustering

In [5]:
# # classic-frost-1096 (https://wandb.ai/eth-cosmo/y3-deep-lss/runs/fp2vxm07/overview)
# # first v14 clustering probes run
# model_dir = "/pscratch/sd/a/athomsen/run_files/v14/extended/clustering/mutual_info/2025-05-14_23-10-45_deepsphere_default"
# # n_steps = 80_000
# # n_steps = 160_000
# n_steps = 240_000

# checkpoint_number = n_steps // 10_000
# params = ["Om", "s8", "w0", "bg1", "bg2", "bg3", "bg4"]
# n_z = 4
# clustering = True

### combined

In [6]:
# grateful-universe-1093 (https://wandb.ai/eth-cosmo/y3-deep-lss/runs/tun9hdvl/overview)
# mythical-cantina-1094 (https://wandb.ai/eth-cosmo/y3-deep-lss/runs/6w6yju8t/overview)
# first v14 combined probes run
model_dir = "/pscratch/sd/a/athomsen/run_files/v14/extended/combined/mutual_info/2025-04-30_02-27-42_deepsphere_default"
# n_steps = 100_000
# n_steps = 200_000
# n_steps = 300_000
n_steps = 400_000

checkpoint_number = n_steps // 10_000
params = ["Om", "s8", "w0", "Aia", "n_Aia", "bta", "bg1", "bg2", "bg3", "bg4"]
n_z = 8
combined = True

# v15

### lensing

In [7]:
# # likely-vortex-1179 (https://wandb.ai/eth-cosmo/y3-deep-lss/runs/9d63ista/overview)
# model_dir = "/pscratch/sd/a/athomsen/run_files/v15/extended/lensing/mutual_info/default/v2"
# # n_steps = 200_000
# n_steps = 400_000
# # n_steps = 600_000

# checkpoint_number = n_steps // 10_000
# params = ["Om", "s8", "w0", "Aia", "n_Aia", "bta"]
# n_z = 4
# lensing = True

### combined

In [8]:
# # deep-sound-1177 (https://wandb.ai/eth-cosmo/y3-deep-lss/runs/kqj560dy/overview)
# model_dir = "/pscratch/sd/a/athomsen/run_files/v15/extended/combined/mutual_info/default/v1"
# # n_steps = 330_000
# n_steps = 440_000

# checkpoint_number = n_steps // 10_000
# params = ["Om", "s8", "w0", "Aia", "n_Aia", "bta", "bg1", "bg2", "bg3", "bg4"]
# n_z = 8
# combined = True

# build the network

In [9]:
base_dir = ""

with open(os.path.join(model_dir, "configs.yaml"), "r") as f:
    net_conf, dlss_conf, msfm_conf = list(yaml.load_all(f, Loader=yaml.FullLoader))
data_vec_pix = files.load_pixel_file(msfm_conf)[0]

smoothing_kwargs = configuration.get_smoothing_kwargs(
    net_conf["run"]["loss_func"], msfm_conf, dlss_conf, net_conf, dir_base="."
)
smoothing_kwargs["max_batch_size"] = 1

# size of the summary statistic, depends on the loss function
out_features =  2 * len(params)
# out_features =  len(params)

# load the layers
network = NETWORKS[net_conf["network"]["name"]](
    out_features=out_features, smoothing_kwargs=smoothing_kwargs, **net_conf["network"]["kwargs"]
).get_layers()

# build the model, same regardless of the loss function (fiducial or grid)
model = BaseModel(
    network=network,
    n_side=msfm_conf["analysis"]["n_side"],
    indices=data_vec_pix,
    n_neighbors=net_conf["network"]["n_neighbors"],
    input_shape=(None, len(data_vec_pix), n_z),
    checkpoint_dir=os.path.join(base_dir, model_dir, "checkpoint"),
    max_batch_size=64,
    restore_checkpoint=False,
    strategy=None,
)

# # for manual control over the checkpoint, to match the evaluated grid
model.restore_model_from_checkpoint_path(
    os.path.join(base_dir, model_dir, "checkpoint", f"ckpt-{checkpoint_number}")
)

2025-09-16 02:15:23.599608: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 37794 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-40GB, pci bus id: 0000:41:00.0, compute capability: 8.0


Using the per channel smoothing repetitions [  6   3   2   1 107  50  30  21]
Using the per channel smoothing scales sigma = [13.11  9.27  7.57  5.35 55.35 37.84 29.31 24.52] arcmin, fwhm = [ 30.86  21.82  17.82  12.6  130.34  89.1   69.01  57.74] arcmin
Successfully loaded sparse kernel indices and values from ./smoothing
Successfully created the sparse kernel tensor
Adding white noise with sigma [0.03520349 0.02952409 0.02803563 0.02272882 0.85135    0.34375
 0.25989    0.24127   ] to the smoothed map. Note that networks outputs are nondeterministic, even for training=False
25-09-16 02:15:24 regression_h INF   Using a dense regression head 
25-09-16 02:15:24 regression_h WAR   Using dropout with probability 0.01 in the regression head 
25-09-16 02:15:24 base_model.p INF   Initializing with a HealpyGCNN model 
Detected a reduction factor of 32.0, the input with nside 512 will be transformed to 16 during a forward pass. Checking for consistency with indices...
indices seem consistent..

In [10]:
out_file = os.path.join(base_dir, model_dir, f"preds_{n_steps}.h5")
print("out_file =", out_file, "\n")

# as a diagnostic
_, grid_preds, grid_cosmos, file_dict = preprocessing.get_reshaped_network_preds(
    base_dir,
    model_dir,
    n_steps,
    with_fidu=False,
)
rng = np.random.default_rng(12)
i_rand = rng.integers(0, grid_preds.shape[0], 1_000)

out_file = /pscratch/sd/a/athomsen/run_files/v14/extended/combined/mutual_info/2025-04-30_02-27-42_deepsphere_default/preds_400000.h5 

25-09-16 02:15:27 input_output INF   Loading predictions from /pscratch/sd/a/athomsen/run_files/v14/extended/combined/mutual_info/2025-04-30_02-27-42_deepsphere_default/preds_400000.h5 
25-09-16 02:15:27 input_output INF   Array shapes: 
25-09-16 02:15:27 input_output INF   fiducial/vali/pred = (40000, 20) 
25-09-16 02:15:27 input_output INF   fiducial/vali/i_example = (40000,) 
25-09-16 02:15:27 input_output INF   fiducial/vali/i_noise = (40000,) 
25-09-16 02:15:27 input_output INF   grid/pred          = (2500, 80, 20) 
25-09-16 02:15:27 input_output INF   grid/cosmo         = (2500, 80, 10) 
25-09-16 02:15:27 input_output INF   grid/i_example     = (2500, 80) 
25-09-16 02:15:27 input_output INF   grid/i_noise       = (2500, 80) 
25-09-16 02:15:27 input_output INF   grid/i_sobol       = (2500, 80) 


25-09-16 02:15:27 preprocessin INF   Shapes after c

# CosmoGrid mocks

### choose mock

In [21]:
base_dir = "/pscratch/sd/a/athomsen/v11desy3/v14/extended/obs"
# base_dir = "/pscratch/sd/a/athomsen/v11desy3/v15/extended/obs"

noise_file = os.path.join(base_dir, "fiducial_bench_obs_maps_noise.h5")

mock_labels = []
mock_files = []

# # benchmark
# mock_labels.append("bench_fidu")
# mock_files.append(os.path.join(base_dir, "fiducial_bench_obs_maps.h5"))

# mock_labels.append("bench_box")
# mock_files.append(os.path.join(base_dir, "box_size_obs_maps.h5"))

# mock_labels.append("bench_particle")
# mock_files.append(os.path.join(base_dir, "particle_count_obs_maps.h5"))

# mock_labels.append("bench_redshift")
# mock_files.append(os.path.join(base_dir, "redshift_resolution_obs_maps.h5"))

# # baryons
# mock_labels.append("bench_fidu_dmo")
# mock_files.append(os.path.join(base_dir, "fiducial_bench_obs_maps_dmo.h5"))

# intrinsic alignments
mock_labels.append("ia_shell")
mock_files.append("/pscratch/sd/a/athomsen/v11desy3/v15/extended/obs/fiducial_bench_obs_maps_Aia=0.5,eta=1_shell.h5")

# # source clustering
# mock_labels.append("source_clustering_bgs_low")
# mock_files.append(os.path.join(base_dir, "fiducial_bench_obs_maps_source_clustering_bgs_low.h5"))

# mock_labels.append("source_clustering_bgs_high")
# mock_files.append(os.path.join(base_dir, "fiducial_bench_obs_maps_source_clustering_bgs_high.h5"))


In [22]:
plot_diagnostics = False
separate_noise = False

if separate_noise:
    with h5py.File(noise_file, "r") as f_in:
        noise_maps = f_in["obs/maps"][:]

for mock_label, mock_file in zip(mock_labels, mock_files):
    print(mock_label)

    if separate_noise:
        mock_file = mock_file[:-3] + "_noiseless.h5"
    
    with h5py.File(mock_file, "r") as f_in:
        mock_maps = f_in["obs/maps"][:]
        print("mock_maps.shape =", mock_maps.shape)

    if separate_noise:
        mock_maps += noise_maps
    
    # normalize
    if lensing or combined:
        mock_maps[:,:,:4] /= msfm_conf["analysis"]["normalization"]["lensing"]
    if clustering or combined:
        mock_maps[:,:,4:] /= msfm_conf["analysis"]["normalization"]["clustering"]

    if mock_maps.shape[-1] == 8:
        if lensing:
            mock_maps = mock_maps[:,:,:4]
        if clustering:
            mock_maps = mock_maps[:,:,4:]
    
    batch_size = 10
    mock_preds = []
    for start in tqdm(range(0, mock_maps.shape[0], batch_size)):
        end = start + batch_size
        mock_preds.append(model(mock_maps[start:end], training=False).numpy())
    mock_preds = np.concatenate(mock_preds, axis=0)
    assert mock_preds.shape[0] == mock_maps.shape[0]
    print("mock_preds.shape =", mock_preds.shape)
    
    # mock_preds = model(mock_maps, training=False).numpy()
    
    if plot_diagnostics:
        tri = TriangleChain()
        tri.scatter(
            np.array(grid_preds)[i_rand], 
            scatter_kwargs={"s": 10, "marker": "o"}
        )
        tri.scatter(
            mock_preds, 
            scatter_kwargs={"s": 10, "marker": "*"},
            color="k",
            scatter_vline_1D=True,
            plot_histograms_1D=False,
        )
    
    
    out_label = f"mocks/pred/{mock_label}"
    with h5py.File(out_file, "a") as f_out:
        if out_label in f_out:
            del f_out[out_label]
        f_out.create_dataset(name=out_label, data=mock_preds)

ia_shell
mock_maps.shape = (80, 458752, 8)


100%|██████████| 8/8 [00:07<00:00,  1.03it/s]

mock_preds.shape = (80, 20)


In [ ]:
stop

# forward model the external mocks

In [ ]:
def evaluate_mock(out_label, lensing_file=None, clustering_file=None, nest_in=False, plot_diagnostics=False):
    obs_label = os.path.split(out_label)[1]
    
    # load the mock
    if lensing_file is not None:
        with h5py.File(lensing_file, "r") as f_in:
            gamma1 = []
            gamma2 = []
            for j in range(1,5):
                gamma1.append(f_in[f"metacal/raw_gamma1_bin{j}"])
                gamma2.append(f_in[f"metacal/raw_gamma2_bin{j}"])
            gamma1 = np.stack(gamma1, axis=-1)
            gamma2 = np.stack(gamma2, axis=-1)
            
            wl_map = np.stack([gamma1, gamma2], axis=-1)

        if plot_diagnostics:
            hp.mollview(gamma1[:,0], nest=nest_in, title="input gamma1")
            hp.mollview(gamma2[:,0], nest=nest_in, title="input gamma2")
    else:
        wl_map = None

    if clustering_file is not None:
        with h5py.File(clustering_file, "r") as f_in:
            gc_map = []
            for i in range(1,5):
                gc_map.append(f_in[f"maglim/galaxy_counts_bin{i}"][:])
            gc_map = np.stack(gc_map, axis=-1)

        if plot_diagnostics:
            hp.mollview(gc_map[:,0], nest=nest_in, title="input galaxy counts")
    else:
        gc_map = None

    # forward model
    obs_map, _, _ = observation.forward_model_observation_map(
        wl_gamma_map=wl_map,
        gc_count_map=gc_map,
        conf=msfm_conf,
        apply_norm=True,
        with_padding=True,
        nest_in=nest_in,
        # add_multiplicative_shear_bias=True,
    )

    # evaluate
    obs_pred = model(obs_map[np.newaxis], training=False)
    obs_pred = np.squeeze(obs_pred)
    print(out_label, ": obs_pred =", obs_pred)

    if plot_diagnostics:
        tri = TriangleChain()
        tri.scatter(
            np.array(grid_preds)[i_rand], 
            scatter_kwargs={"s": 10, "marker": "o"}
        )
        tri.scatter(
            np.array(obs_pred)[np.newaxis], 
            scatter_kwargs={"s": 500, "marker": "*"},
            color="k",
            scatter_vline_1D=True,
            plot_histograms_1D=False,
        )
        if lensing_file is not None and clustering_file is None:
            probe = "lensing"
        elif lensing_file is None and clustering_file is not None:
            probe = "clustering"
        elif lensing_file is not None and clustering_file is not None:
            probe = "combined"
        title = f"{probe}_{obs_label}"
        tri.fig.suptitle(title, fontsize=48, y=0.9)
        tri.fig.savefig(f"./plots/summary_{title}.png", bbox_inches="tight", dpi=50)

    # save
    with h5py.File(out_file, "a") as f_out:
        if out_label in f_out:
            del f_out[out_label]
        f_out.create_dataset(name=out_label, data=obs_pred)


## Buzzard

### weak lensing

In [ ]:
if lensing:    
    buzzard_indices = list(range(0, 16))
    buzzard_indices.remove(1)
    # buzzard_indices = [0]
    for i in buzzard_indices:
        # out_label = f"mocks/pred/Buzzard_{i}"
        # lensing_file = f"/pscratch/sd/j/jbucko/DESY3/mock_observations/lensing/buzzard_flock/{i}/DESY3_mock_observation_buzzard_flock_v14_shear_noise+WL.h5"
        
        # out_label = f"mocks/pred/Buzzard_42_{i}"
        # lensing_file = f"/pscratch/sd/j/jbucko/DESY3/mock_observations/lensing/buzzard_flock/{i}/DESY3_mock_observation_buzzard_flock_v14_shear_noise+WL_iseed_42.h5"

        # out_label = f"mocks/pred/Buzzard_289_{i}"
        # lensing_file = f"/pscratch/sd/j/jbucko/DESY3/mock_observations/lensing/buzzard_flock/{i}/DESY3_mock_observation_buzzard_flock_v14_shear_noise+WL_iseed_289.h5"

        out_label = f"mocks/pred/Buzzard_{i}"
        lensing_file = f"/pscratch/sd/j/jbucko/DESY3/mock_observations/lensing/buzzard_flock/{i}/DESY3_mock_observation_buzzard_flock_v14_shear_noise+WL_iseed_42_varied.h5"

        evaluate_mock(
            out_label, 
            lensing_file=lensing_file, 
            nest_in=False, 
            # plot_diagnostics=True
        )

### galaxy clustering

In [ ]:
if clustering:
    I = [0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
    J = [0, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 8, 8, 11, 11]
    K = ["a"] + 7 * ["a", "b"]

    # I = [I[0]]
    # J = [J[0]]
    # K = [K[0]]

    for i, j, k in zip(I, J, K):
        out_label = f"mocks/pred/Buzzard_{i}"
        clustering_file = f"/pscratch/sd/j/jbucko/DESY3/mock_observations/lensing/buzzard_flock/{i}/DESY3_mock_observation_Buzzard_{j}_Y3{k}.h5"

        evaluate_mock(
            out_label,
            clustering_file=clustering_file, 
            nest_in=False, 
            # plot_diagnostics=True,
            plot_diagnostics=False,
        )

### combined probes

In [ ]:
if combined:
    # buzzard_indices = list(range(0, 15))
    # buzzard_indices.remove(1)

    I = [0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
    J = [0, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 8, 8, 11, 11]
    K = ["a"] + 7 * ["a", "b"]

    # I = [I[0]]
    # J = [J[0]]
    # K = [K[0]]

    for i, j, k in zip(I, J, K):
        out_label = f"mocks/pred/Buzzard_{i}"
        lensing_file = f"/pscratch/sd/j/jbucko/DESY3/mock_observations/lensing/buzzard_flock/{i}/DESY3_mock_observation_buzzard_flock_v14_shear_noise+WL_iseed_42_varied.h5"
        clustering_file = f"/pscratch/sd/j/jbucko/DESY3/mock_observations/lensing/buzzard_flock/{i}/DESY3_mock_observation_Buzzard_{j}_Y3{k}.h5"

        evaluate_mock(
            out_label,
            lensing_file=lensing_file,
            clustering_file=clustering_file, 
            nest_in=False, 
            # plot_diagnostics=True,
            plot_diagnostics=False,
        )

In [ ]:
# if combined:
#     # buzzard_indices = list(range(0, 15))
#     # buzzard_indices.remove(1)
#     I = [0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
    
#     for i, j, k in zip([0, 2, 3, 4], [0, 3, 3, 4], ["a", "a", "b", "a"]):
#         out_label = f"mocks/pred/Buzzard_{i}"
#         lensing_file = f"/pscratch/sd/j/jbucko/DESY3/mock_observations/lensing/buzzard_flock/{i}/DESY3_mock_observation_buzzard_flock_v14_shear_noise+WL.h5"
#         clustering_file = f"/pscratch/sd/j/jbucko/DESY3/mock_observations/lensing/buzzard_flock/{i}/DESY3_mock_observation_Buzzard_{j}_Y3{k}.h5"

#         evaluate_mock(
#             out_label,
#             lensing_file=lensing_file,
#             clustering_file=clustering_file, 
#             nest_in=False, 
#             plot_diagnostics=True,
#             # plot_diagnostics=False,
#         )

In [ ]:
# if combined:
#     # buzzard_indices = list(range(0, 15))
#     # buzzard_indices.remove(1)
#     buzzard_indices = [0, 2, 3, 4]
#     for i in buzzard_indices:
#         out_label = f"mocks/pred/Buzzard_{i}"
#         lensing_file = f"/pscratch/sd/j/jbucko/DESY3/mock_observations/lensing/buzzard_flock/{i}/DESY3_mock_observation_buzzard_flock_v14_shear_noise+WL.h5"
#         clustering_file = f"/pscratch/sd/j/jbucko/DESY3/mock_observations/lensing/buzzard_flock/{i}/DESY3_mock_observation_Buzzard_0_Y3a.h5"
        
#         evaluate_mock(
#             out_label,
#             lensing_file=lensing_file,
#             clustering_file=clustering_file, 
#             nest_in=False, 
#             # plot_diagnostics=True,
#             plot_diagnostics=False,
#         )

# OLD

## old

In [ ]:
# # base_dir = "/pscratch/sd/a/athomsen/v11desy3/v14/extended/obs"
# base_dir = "/pscratch/sd/a/athomsen/v11desy3/v15/extended/obs"

# mock_labels = []
# mock_files = []

# # mock_labels.append("fiducial")
# # mock_files.append(os.path.join(base_dir, "cosmo_fiducial_obs_maps.h5"))

# # mock_labels.append("cosmo_114996")
# # mock_files.append(f"/pscratch/sd/a/athomsen/v11desy3/v14/extended/obs/{mock_label}_obs_maps.h5")

# # benchmark
# # mock_labels.append("bench_fidu")
# # mock_files.append(os.path.join(base_dir, "fiducial_bench_obs_maps.h5"))

# # mock_labels.append("bench_box")
# # mock_files.append(os.path.join(base_dir, "box_size_obs_maps.h5"))

# # mock_labels.append("bench_particle")
# # mock_files.append(os.path.join(base_dir, "particle_count_obs_maps.h5"))

# # mock_labels.append("bench_redshift")
# # mock_files.append(os.path.join(base_dir, "redshift_resolution_obs_maps.h5"))

# # # source clustering
# # mock_labels.append("source_clustering_bgs_low")
# # mock_files.append(os.path.join(base_dir, "fiducial_bench_obs_maps_source_clustering_bgs_low.h5"))

# # mock_labels.append("source_clustering_bgs_high")
# # mock_files.append(os.path.join(base_dir, "fiducial_bench_obs_maps_source_clustering_bgs_high.h5"))

# # intrinsic alignment
# # mock_labels.append("Aia=0")
# # mock_files.append(os.path.join(base_dir, "cosmo_fiducial_obs_maps_no_ia.h5"))

# # mock_labels.append("Aia=1")
# # mock_files.append(os.path.join(base_dir, "cosmo_fiducial_obs_maps_ia.h5"))

# # mock_labels.append("fidu_bary")
# # mock_files.append(os.path.join(base_dir, "cosmo_fiducial_obs_maps_bary.h5"))

# # mock_labels.append("fidu_dmo")
# # mock_files.append(os.path.join(base_dir, "cosmo_fiducial_obs_maps_no_bary.h5"))

# # mock_labels.append("fidu_dmo")
# # mock_files.append(os.path.join(base_dir, "cosmo_fiducial_obs_maps_no_bary.h5"))

# # mock_labels.append("bench_fidu_seed=1234")
# # mock_files.append(os.path.join(base_dir, "fiducial_bench_obs_maps_seed=1234.h5"))

# # mock_labels.append("bench_fidu_dmo")
# # mock_files.append(os.path.join(base_dir, "fiducial_bench_obs_maps_dmo.h5"))

# # mock_labels.append("reduced_shear")
# # mock_files.append(os.path.join(base_dir, "fiducial_bench_obs_maps_reduced_shear.h5"))

# # # intrinsic alignment
# # mock_labels.append("cosmo_114996_Aia=0")
# # mock_files.append(os.path.join(base_dir, "cosmo_114996_obs_maps_Aia=0.h5"))

# # mock_labels.append("cosmo_114996_Aia=1")
# # mock_files.append(os.path.join(base_dir, "cosmo_114996_obs_maps_Aia=1.h5"))

# # mock_labels.append("cosmo_114996_Aia=-1")
# # mock_files.append(os.path.join(base_dir, "cosmo_114996_obs_maps_Aia=-1.h5"))

# mock_labels.append("ia_bin")
# mock_files.append(os.path.join(base_dir, "fiducial_bench_obs_maps_eta_ia=1_bin.h5"))

# mock_labels.append("ia_shell")
# mock_files.append(os.path.join(base_dir, "fiducial_bench_obs_maps_eta_ia=1_shell.h5"))



### evaluate

In [ ]:
# plot_diagnostics = False

# for mock_label, mock_file in zip(mock_labels, mock_files):
#     print(mock_label)
    
#     with h5py.File(mock_file, "r") as f_in:
#         mock_maps = f_in["obs/maps"][:]
#         print("mock_maps.shape =", mock_maps.shape)
    
#     # normalize
#     if lensing or combined:
#         mock_maps[:,:,:4] /= msfm_conf["analysis"]["normalization"]["lensing"]
#     if clustering or combined:
#         mock_maps[:,:,4:] /= msfm_conf["analysis"]["normalization"]["clustering"]

#     if mock_maps.shape[-1] == 8:
#         if lensing:
#             mock_maps = mock_maps[:,:,:4]
#         if clustering:
#             mock_maps = mock_maps[:,:,4:]
    
#     batch_size = 10
#     mock_preds = []
#     for start in tqdm(range(0, mock_maps.shape[0], batch_size)):
#         end = start + batch_size
#         mock_preds.append(model(mock_maps[start:end], training=False).numpy())
#     mock_preds = np.concatenate(mock_preds, axis=0)
#     assert mock_preds.shape[0] == mock_maps.shape[0]
#     print("mock_preds.shape =", mock_preds.shape)
    
#     # mock_preds = model(mock_maps, training=False).numpy()
    
#     if plot_diagnostics:
#         tri = TriangleChain()
#         tri.scatter(
#             np.array(grid_preds)[i_rand], 
#             scatter_kwargs={"s": 10, "marker": "o"}
#         )
#         tri.scatter(
#             mock_preds, 
#             scatter_kwargs={"s": 10, "marker": "*"},
#             color="k",
#             scatter_vline_1D=True,
#             plot_histograms_1D=False,
#         )
    
    
#     out_label = f"mocks/pred/{mock_label}"
#     with h5py.File(out_file, "a") as f_out:
#         if out_label in f_out:
#             del f_out[out_label]
#         f_out.create_dataset(name=out_label, data=mock_preds)

In [ ]:
# temp_map = np.zeros(hp.nside2npix(512))
# temp_map[data_vec_pix] = mock_maps[0,:,0]

# hp.mollview(temp_map, nest=True)
# hp.gnomview(temp_map, nest=True, rot=(90,-10,0))

In [ ]:
# plot_diagnostics = True

# with h5py.File(mock_file, "r") as f_in:
#     mock_maps = f_in["obs/maps"][:]
#     print("mock_maps.shape =", mock_maps.shape)

# # normalize
# if lensing or combined:
#     mock_maps[:,:,:4] /= msfm_conf["analysis"]["normalization"]["lensing"]
# if clustering or combined:
#     mock_maps[:,:,4:] /= msfm_conf["analysis"]["normalization"]["clustering"]

# if lensing:
#     mock_maps = mock_maps[:,:,:4]
# if clustering:
#     mock_maps = mock_maps[:,:,4:]

# batch_size = 10
# mock_preds = []
# for start in tqdm(range(0, mock_maps.shape[0], batch_size)):
#     end = start + batch_size
#     mock_preds.append(model(mock_maps[start:end], training=False).numpy())
# mock_preds = np.concatenate(mock_preds, axis=0)
# assert mock_preds.shape[0] == mock_maps.shape[0]
# print("mock_preds.shape =", mock_preds.shape)

# # mock_preds = model(mock_maps, training=False).numpy()

# if plot_diagnostics:
#     tri = TriangleChain()
#     tri.scatter(
#         np.array(grid_preds)[i_rand], 
#         scatter_kwargs={"s": 10, "marker": "o"}
#     )
#     tri.scatter(
#         mock_preds, 
#         scatter_kwargs={"s": 10, "marker": "*"},
#         color="k",
#         scatter_vline_1D=True,
#         plot_histograms_1D=False,
#     )


# out_label = f"mocks/pred/{mock_label}"
# with h5py.File(out_file, "a") as f_out:
#     if out_label in f_out:
#         del f_out[out_label]
#     f_out.create_dataset(name=out_label, data=mock_preds)

In [ ]:
# data_vec_pix, patches_pix_dict, _, _ = files.load_pixel_file(msfm_conf)
# n_pix = hp.nside2npix(512)
# i_example = 2

# for i in range(mock_maps.shape[-1]):
#     example_map = np.zeros(n_pix)
#     example_map[data_vec_pix] = mock_maps[i_example,:,i]
#     hp.mollview(example_map, title=f"bin {i}", nest=True)

### check against the CosmoGrid internal predictions

In [ ]:
# # get cosmogrid cosmologies
# data = h5py.File('/global/cfs/cdirs/des/cosmogrid/CosmoGridV1_metainfo.h5','r')
# cosmo_arr = data['parameters/grid'][()]

# grid_cosmo_paths = [item.astype('str').split('/')[-2] for item in cosmo_arr['path_par']]
# cosmo_params_cosmogridv11 = np.c_[cosmo_arr['Om'].astype('float'),cosmo_arr['s8'].astype('float'),cosmo_arr['w0'].astype('float'),cosmo_arr['wa'].astype('float'),cosmo_arr['Ob'].astype('float'),cosmo_arr['ns'].astype('float'),cosmo_arr['H0'].astype('float')/100]

# # select some cosmologies
# cosmogrid_grid_idx = [0,38,654,710]

# # cosmo_params_cosmogridv11[cosmogrid_grid_idx]

In [ ]:
# clustering_file = "/pscratch/sd/j/jbucko/DESY3/v13/linear_bias/clustering/maps/maps_for_compression_v13_clustering_True_lensing_False_grid_cosmo.pkl"
# with open(clustering_file, 'rb') as f:
#     jozef_maps = pickle.load(f)

# n_side = msfm_conf["analysis"]["n_side"]
# n_pix = msfm_conf["analysis"]["n_pix"]
# data_vec_pix = files.load_pixel_file(msfm_conf)[0]
# hp_datapath = "/global/u2/a/athomsen/multiprobe-simulation-forward-model/data/healpy_data"

# with h5py.File(out_file, "a") as f_out:
#     for i in range(jozef_maps.shape[0]):
#         j = cosmogrid_grid_idx[i]
        
#         gc_map = jozef_maps[i]
#         print(gc_map.shape)

#         obs_pred = model(gc_map, training=False)
#         obs_pred = np.squeeze(obs_pred)

#         # print("\n", obs_label, obs_pred, "\n")

#         out_label = f"mocks/pred/grid_{j}" 
#         print(out_label)

#         if out_label in f_out:
#             del f_out[out_label]
#         f_out.create_dataset(name=out_label, data=obs_pred)



In [ ]:
# n_noise = 8
# obs_label = "Cardinal"

# out_file = os.path.join(base_dir, model_dir, f"preds_{n_steps}.h5")
# with h5py.File(out_file, "a") as f_out:
#     obs_file = f"/global/u2/a/athomsen/multiprobe-simulation-forward-model/data/mock_observations/DESY3_mock_observation_{obs_label}.h5"

#     for i in range(n_noise):
#         evaluate_mock(f_out, obs_file, f"Cardinal_noise_{i}")

#     print(f"Saved the prediction to {out_file}")